# SHAP por dentro — verificando as afirmações da aula

Companheiro técnico do walkthrough. Quatro medições sobre o TreeSHAP
nativo do XGBoost:

1. **exatidão, determinismo e o axioma dummy** — eficiência no teste
   inteiro, três refits bit-idênticos, e o teste do dummy (vácuo ou não,
   dito);
2. **o base value** — de onde todo waterfall parte, contra a definição
   de ganho do cap. 17;
3. **interações** — a matriz 41×41 soma de volta às contribuições, e
   nomeia o par que o dependence plot do walkthrough colore;
4. **o custo do interventional** — erro de Monte Carlo do estimador de
   Štrumbelj & Kononenko (2014) contra o exato de caminho.

In [1]:
%pip install -q xgboost scikit-learn scipy matplotlib numpy pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgblib
from scipy.special import expit

pd.set_option("display.width", 120)
RANDOM_STATE = 42

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "README.md").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "README.md").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_60_model as M

FIGDIR = pathlib.Path("figures_generated")
FIGDIR.mkdir(exist_ok=True)

g = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
xgb, _logit = M.fit_models(g)
paciente = M.pick_exemplar(xgb, g)
booster = xgb.get_booster()

AZUL, VERMELHO, CINZA = "#4878a8", "#d1654e", "#9a9a9a"


def dmatrix(frame):
    x, _, _ = M.design_matrix(frame)
    return xgblib.DMatrix(x, enable_categorical=True)


teste = g[g["split"] == "test"].reset_index(drop=True)
dm_teste = dmatrix(teste)
contrib = booster.predict(dm_teste, pred_contribs=True)
margem = booster.predict(dm_teste, output_margin=True)
NOMES = list(M.FEATURES) + ["base"]

desvio = float(np.abs(contrib.sum(axis=1) - margem).max())
print(f"contribuições no teste: {contrib.shape[0]:,} × {contrib.shape[1]} "
      f"(40 features + base)".replace(",", "."))
print(f"eficiência, verificada: |soma das contribuições − margem| máximo = {desvio:.2e}")
print(f"base value (margem): {contrib[0, -1]:.4f} — sigmoide {expit(contrib[0, -1]):.4f} "
      f"(a taxa de óbito do treino é 0,3136 — internals §2)")

ix = teste.index[teste["gold_id"] == int(paciente["gold_id"])][0]
c_pac = contrib[ix]
print(f"paciente-regra gold_id {int(paciente['gold_id'])}: soma + base = "
      f"{float(c_pac.sum()):+.4f} (margem) -> sigmoide {expit(float(c_pac.sum())):.4f} "
      f"= p do modelo {paciente['p_obito']:.4f}")

contribuições no teste: 16.142 × 41 (40 features + base)
eficiência, verificada: |soma das contribuições − margem| máximo = 1.05e-05
base value (margem): -0.7881 — sigmoide 0.3126 (a taxa de óbito do treino é 0,3136 — internals §2)
paciente-regra gold_id 1276776: soma + base = +0.0000 (margem) -> sigmoide 0.5000 = p do modelo 0.5000


## §1 — Exato, determinístico, e o dummy

Duas propriedades que nenhum método amostrado do curso tem — e um axioma
que dá para tentar falsificar no modelo real.

In [3]:
dev_total = float(np.abs(contrib.sum(axis=1) - margem).max())
print(f"eficiência no teste inteiro ({len(contrib):,} linhas): "
      f"desvio máximo {dev_total:.2e}".replace(",", "."))

refits = []
for r in range(3):
    m_r, _ = M.fit_models(g)
    c_r = m_r.get_booster().predict(dm_teste, pred_contribs=True)
    refits.append(c_r)
d01 = float(np.abs(refits[0] - refits[1]).max())
d02 = float(np.abs(refits[0] - refits[2]).max())
print(f"3 refits, mesmas contribuições: desvio máx {max(d01, d02):.2e}")

# axioma dummy: uma feature que nenhuma árvore usa deve ter phi = 0 exato
usadas = set(booster.get_score(importance_type="weight"))
nunca = [c for c in M.FEATURES if c not in usadas]
if nunca:
    idx = [NOMES.index(c) for c in nunca]
    print(f"axioma dummy: {len(nunca)} feature(s) nunca usadas em divisão "
          f"({', '.join(nunca)}); max|φ| nelas = "
          f"{float(np.abs(contrib[:, idx]).max()):.2e}")
else:
    print("axioma dummy: todas as 40 features aparecem em alguma divisão do")
    print("modelo — o teste é vácuo neste fit, e fica dito em vez de omitido.")
print("=> determinístico e exato: zero variância de semente, zero amostragem —")
print("   as duas coisas que nenhum método amostrado do curso pôde prometer.")

eficiência no teste inteiro (16.142 linhas): desvio máximo 1.05e-05


3 refits, mesmas contribuições: desvio máx 0.00e+00
axioma dummy: todas as 40 features aparecem em alguma divisão do
modelo — o teste é vácuo neste fit, e fica dito em vez de omitido.
=> determinístico e exato: zero variância de semente, zero amostragem —
   as duas coisas que nenhum método amostrado do curso pôde prometer.


## §2 — O base value

`pred_contribs` devolve 41 colunas; a última é o ponto de partida de
toda explicação.

In [4]:
x_tr, _, y_tr = M.design_matrix(g[g["split"] == "train"])
m_tr = booster.predict(xgblib.DMatrix(x_tr, enable_categorical=True),
                       output_margin=True)
base = float(contrib[0, -1])
print(f"base value: {base:+.4f} (margem)")
print(f"margem média do treino: {m_tr.mean():+.4f}")
print(f"sigmoide(base) = {expit(base):.4f} | taxa de óbito do treino: "
      f"{y_tr.mean():.4f}")
print()
print("o base é a predição de um paciente sobre o qual nada se sabe — e mora")
print("ao lado da prevalência do treino. Todo waterfall parte daqui; o cap. 17")
print("define o ganho do jogo exatamente como f(x) menos esta média.")

base value: -0.7881 (margem)
margem média do treino: -1.0014
sigmoide(base) = 0.3126 | taxa de óbito do treino: 0.3136

o base é a predição de um paciente sobre o qual nada se sabe — e mora
ao lado da prevalência do treino. Todo waterfall parte daqui; o cap. 17
define o ganho do jogo exatamente como f(x) menos esta média.


## §3 — As interações somam de volta

`pred_interactions` decompõe cada contribuição em efeitos principais +
pares. A verificação de consistência é a soma; o subproduto é saber qual
par o modelo mais usa em conjunto — que é o que o dependence plot do
walkthrough (§5) mostra em cor.

In [5]:
sub = np.arange(2000)
inter = booster.predict(xgblib.DMatrix(M.design_matrix(teste.iloc[sub])[0],
                                       enable_categorical=True),
                        pred_interactions=True)
print(f"matriz de interações: {inter.shape} (2.000 linhas × 41 × 41)")
soma_linhas = inter.sum(axis=2)
dev_i = float(np.abs(soma_linhas - contrib[sub]).max())
print(f"soma das interações de cada feature == contribuição: desvio máx {dev_i:.2e}")
par = np.abs(inter[:, :-1, :-1]).mean(axis=0)
np.fill_diagonal(par, 0.0)
i, j = np.unravel_index(np.argmax(par), par.shape)
print(f"o par que mais interage no teste: {NOMES[i]} × {NOMES[j]} "
      f"(média|φ_ij| = {par[i, j]:.4f})")
print("=> é a interação que o dependence plot do walkthrough §5 mostra em cor.")

matriz de interações: (2000, 41, 41) (2.000 linhas × 41 × 41)
soma das interações de cada feature == contribuição: desvio máx 6.20e-06
o par que mais interage no teste: idade_anos × meses_desde_mar2020 (média|φ_ij| = 0.1904)
=> é a interação que o dependence plot do walkthrough §5 mostra em cor.


## §4 — O custo do interventional

O walkthrough §6 estimou Shapley por permutação com 60 amostras. Aqui, o
erro dessa estimativa em função do orçamento, contra o exato de caminho —
lembrando que os dois **não convergem um para o outro**: respondem
condicionais diferentes.

In [6]:
train_pool = g[g["split"] == "train"].reset_index(drop=True)
fundo = train_pool.sample(200, random_state=RANDOM_STATE).reset_index(drop=True)


def phi_idade(n_perm, semente):
    rng = np.random.default_rng(semente)
    acc = 0.0
    for t in range(n_perm):
        bg = fundo.iloc[int(rng.integers(len(fundo)))]
        ordem_f = list(rng.permutation(M.FEATURES))
        linha = {c: bg[c] for c in g.columns}
        linhas = [dict(linha)]
        for c in ordem_f:
            linha[c] = paciente[c]
            linhas.append(dict(linha))
        frame = pd.DataFrame(linhas).astype(g.dtypes)
        m = booster.predict(dmatrix(frame), output_margin=True)
        k = ordem_f.index("idade_anos")
        acc += float(m[k + 1] - m[k]) / n_perm
    return acc


exato = float(contrib[teste.index[teste["gold_id"] == int(paciente["gold_id"])][0],
                      NOMES.index("idade_anos")])
print(f"pred_contribs de idade (exato, caminho): {exato:+.4f}")
print("permutações   5 estimativas (sementes 0–4)                 dp")
medias = {}
for n in (10, 30, 60):
    ests = [phi_idade(n, s) for s in range(5)]
    medias[n] = float(np.mean(ests))
    print(f"{n:11d}   " + " ".join(f"{e:+.3f}" for e in ests)
          + f"   {np.std(ests):.4f}")
print()
print(f"com 5 repetições o dp ainda oscila — o decaimento √n aparece devagar.")
print(f"E repare no CENTRO: a média com 60 permutações é {medias[60]:+.3f}, perto")
print(f"do interventional do walkthrough §6, ACIMA do exato de caminho "
      f"({exato:+.4f}):")
print("as estimativas não convergem para o pred_contribs — medem outra")
print("condicional. O que não oscila: o exato custa UMA predição, dp zero.")

pred_contribs de idade (exato, caminho): +0.9231
permutações   5 estimativas (sementes 0–4)                 dp


         10   +0.733 +0.899 +0.946 +1.151 +1.084   0.1464


         30   +0.814 +0.752 +1.239 +1.188 +0.973   0.1944


         60   +0.847 +0.986 +1.217 +1.084 +1.049   0.1213

com 5 repetições o dp ainda oscila — o decaimento √n aparece devagar.
E repare no CENTRO: a média com 60 permutações é +1.037, perto
do interventional do walkthrough §6, ACIMA do exato de caminho (+0.9231):
as estimativas não convergem para o pred_contribs — medem outra
condicional. O que não oscila: o exato custa UMA predição, dp zero.


## Summary

TreeSHAP exato: uma predição por explicação, zero sementes, eficiência
em ponto flutuante, dummy testável. O que ele não decide por você: qual
condicional contar (caminho × intervenção) — e essa escolha, como todas
as deste curso, se mede: em contradições fabricadas por coalizão, pela
mesma `gate_impossible` de sempre.